In [2]:
import numpy as np
from utils.config import load_config, Configuration
from utils.utils import (
    retrieve_stacked_betas,
    retrieve_roi_mask,
    filter_roi_mask,
    retrieve_stacked_betas_test,
)
import pandas as pd


config = load_config("config.yaml")

In [4]:
subj = 1
mask_value = 1

samples_per_set = 32



mask = np.load(f"data/mds_dir/subj_{subj:02d}/mask_{mask_value}_averaged_mds.npy")

In [5]:
mask = retrieve_roi_mask(config, subj)
betas_positive, image_ids_positive = retrieve_stacked_betas(config, subj, "averaged", 0, label_subset_name=config.nsd_positive_subset)
betas_negative, image_ids_negative = retrieve_stacked_betas(config, subj, "averaged", 0, label_subset_name=config.nsd_negative_subset)



mask_voxel_indices = filter_roi_mask(mask_value, mask)
mask_voxel_indices = mask_voxel_indices[0]
n_voxels = mask_voxel_indices.shape[0]
n_images = samples_per_set * 2


NameError: name 'retrieve_roi_mask' is not defined

In [ ]:
masked_voxel_betas_positive = betas_positive.T[mask_voxel_indices].T[:samples_per_set]
masked_voxel_betas_negative = betas_negative.T[mask_voxel_indices].T[:samples_per_set]

image_ids_positive = image_ids_positive[:samples_per_set]
image_ids_negative = image_ids_negative[:samples_per_set]

concatenated_voxel_betas = np.concatenate(
    (masked_voxel_betas_positive, masked_voxel_betas_negative), axis=0
)  # shape: (n_images, n_voxels)



# Step 1: Calculate the mean of each cell across all images
cell_means = np.mean(concatenated_voxel_betas, axis=0)



# Step 2: Get the indices that would sort the cells by their mean
sorted_indices = np.argsort(cell_means)

# Step 3: Sort the cells based on the calculated means
concatenated_voxel_betas = concatenated_voxel_betas[:, sorted_indices].T

# Step 4: (Optional) Sort the means themselves for reference
means_over_cells = cell_means[sorted_indices]
means_over_images = np.mean(concatenated_voxel_betas, axis=0)

means_over_images_positive = means_over_images[:samples_per_set]
means_over_images_negative = means_over_images[samples_per_set:]



positive_min_index = np.argmin(means_over_images_positive)
positive_max_index = np.argmax(means_over_images_positive)

negative_min_index = np.argmin(means_over_images_negative)
negative_max_index = np.argmax(means_over_images_negative)


print(f"Positive: max: {means_over_images_positive[positive_max_index]:.02f}\tmin: {means_over_images_positive[positive_min_index]:.02f}")
print(f"Negative: max: {means_over_images_negative[negative_max_index]:.02f}\tmin: {means_over_images_negative[negative_min_index]:.02f}")


from IPython.display import Image, display
import os

display(Image(filename=os.path.join(config.images_target_dir, f"{image_ids_positive[positive_max_index]}.jpg")))
display(Image(filename=os.path.join(config.images_target_dir, f"{image_ids_positive[positive_min_index]}.jpg")))
display(Image(filename=os.path.join(config.images_target_dir, f"{image_ids_negative[negative_max_index]}.jpg")))
display(Image(filename=os.path.join(config.images_target_dir, f"{image_ids_negative[negative_min_index]}.jpg")))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.gridspec as gridspec


heatmap_data = concatenated_voxel_betas  # Replace with your matrix data
mean_response_data = means_over_images  # Replace with your array data

# Set up the figure
fig = plt.figure(figsize=(10, 8))
gs = gridspec.GridSpec(2, 1, height_ratios=[3, 1], hspace=0.3)

# Plot the heatmap
ax1 = plt.subplot(gs[0])
im = ax1.imshow(heatmap_data, aspect='auto', cmap='coolwarm', interpolation='nearest')
ax1.set_xlabel('Image number', fontsize=12)
ax1.set_ylabel('Cell number', fontsize=12)
plt.colorbar(im, ax=ax1, orientation='vertical', label='Response')
ax1.set_title('Heatmap', fontsize=14)

# plt.show()

# Plot the bar chart
ax2 = plt.subplot(gs[1])
ax2.bar(range(len(mean_response_data)), mean_response_data, color='blue', edgecolor='black', width=0.8)
ax2.set_xlabel('Image number', fontsize=12)
ax2.set_ylabel('Mean response', fontsize=12)
ax2.set_title('Mean Response per Image', fontsize=14)

# Add example insets (if needed)
# Example inset positions for visualization
inset_positions = [20, 48]
inset_images = [np.random.rand(10, 10) for _ in inset_positions]  # Replace with actual image data

# for i, pos in enumerate(inset_positions):
#     inset_ax = ax2.inset_axes([pos / 100, 0.6, 0.1, 0.3])  # Adjust these values for positioning
#     inset_ax.imshow(inset_images[i], cmap='gray')
#     inset_ax.axis('off')
#     ax2.annotate('', xy=(pos, mean_response_data[pos]), 
#                  xytext=(pos, 0.5), arrowprops=dict(arrowstyle="->"))

plt.tight_layout()
plt.show()
